# 04 - MTTR before vs. after

Loads baseline and post-system snapshots written by `evaluation/baseline_capture.py` (`data/baseline/`, `data/post_system/`). When those folders are empty the notebook simulates data so the analysis can be developed first.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))   # make the project packages importable from the notebook

In [ ]:
import numpy as np
import pandas as pd

from evaluation.mttr_analyser import compare, compute_ttr, load_snapshots, plot_comparison, summarise_mttr

def simulate(n, median_hours, seed):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        created = pd.Timestamp("2026-01-01", tz="UTC") + pd.Timedelta(hours=float(rng.uniform(0, 600)))
        severity = str(rng.choice(["CRITICAL", "HIGH", "MEDIUM", "LOW"], p=[0.15, 0.3, 0.35, 0.2]))
        ttr = float(rng.lognormal(np.log(median_hours), 0.6))
        rows.append(dict(finding_id=f"f{seed}-{i}", severity=severity, created_at=created, updated_at=created + pd.Timedelta(hours=1), workflow_status="NEW"))
        rows.append(dict(finding_id=f"f{seed}-{i}", severity=severity, created_at=created, updated_at=created + pd.Timedelta(hours=ttr), workflow_status="RESOLVED"))
    return pd.DataFrame(rows)

def load_or_simulate(folder, n, median_hours, seed):
    path = ROOT / "data" / folder
    if path.exists() and any(path.rglob("*.csv")):
        return load_snapshots(path), "real"
    return simulate(n, median_hours, seed), "SIMULATED"

baseline_raw, b_src = load_or_simulate("baseline", 120, 240, 1)
post_raw, p_src = load_or_simulate("post_system", 120, 72, 2)
print("baseline:", b_src, "| post-system:", p_src)

## Time-to-remediate per finding

In [ ]:
baseline_ttr, post_ttr = compute_ttr(baseline_raw), compute_ttr(post_raw)
print(len(baseline_ttr), "baseline and", len(post_ttr), "post-system resolved findings")
baseline_ttr.head()

## MTTR comparison and significance
One-sided Mann-Whitney U (H1: post-system TTR is lower). TTR is right-skewed, so a rank test is used instead of a t-test.

In [ ]:
result = compare(baseline_ttr, post_ttr)
print(f"Mann-Whitney U = {result['u_statistic']:.1f},  one-sided p = {result['p_value']:.3g}")
result["table"].round(1)

In [ ]:
fig = plot_comparison(summarise_mttr(baseline_ttr), summarise_mttr(post_ttr))
fig.show()

## Caveats to report
- `UpdatedAt` on a RESOLVED finding is a proxy for the resolution time.
- Findings never observed as resolved are excluded (right-censoring); report their share too.
- Compare like-for-like finding mixes and periods.

In [ ]:
unresolved = 1 - len(post_ttr) / post_raw["finding_id"].nunique()
print(f"post-system findings not yet resolved: {unresolved:.1%}")